#### 逐日明细表

In [ ]:
-- 逐日明细表
DROP TABLE IF EXISTS xyf_jingying_dev.price20_preroute_cust_lss5;
CREATE TABLE xyf_jingying_dev.price20_preroute_cust_lss5 AS


SELECT  m.pt                                                                                               AS 曝光日期
       ,m.user_no
	   ,c.cust_no                                                                                       
       ,m.风险原始定价
       ,m.飞跃会员类型分组
	   ,m.vip_classify_group
       ,CASE WHEN xyf_dwd.randomv3('jkjzzplk' ,m.user_no ,3) BETWEEN 0 AND 99 THEN '测试组'
             WHEN xyf_dwd.randomv3('jkjzzplk' ,m.user_no ,3) BETWEEN 100 AND 999 THEN '对照组'  ELSE '其他' END AS group_tag
       ,NVL(risk_I20, 0)                                                                                       AS risk_I20
       ,CASE WHEN price_test.debit_amt < 4000 THEN '[0k,4k)'
             WHEN price_test.debit_amt < 5000 THEN '[4k,5k)'
             WHEN price_test.debit_amt < 6000 THEN '[5k,6k)'
             WHEN price_test.debit_amt < 7000 THEN '[6k,7k)'
             WHEN price_test.debit_amt < 8000 THEN '[7k,8k)'
             WHEN price_test.debit_amt < 9000 THEN '[8k,9k)'
             WHEN price_test.debit_amt < 10000 THEN '[9k,10k)'
             WHEN price_test.debit_amt >= 10000 THEN '[10k,+)'  ELSE '其他' END                              AS 额度区间
       ,CASE WHEN price_test.debit_amt >= 4000 AND price_test.debit_amt < 10000 THEN 1  ELSE 0 END           AS 可用额度4k_10k
	   ,feiyue_qianyue.*except(app_user_id)
	   ,lo.*except(user_no, cust_no)
--借款页曝光 
FROM
(
	SELECT  user_no
	       ,date(trigger_timestamp)                    AS pt
	       ,trigger_timestamp                          AS 埋点时间
	       ,GET_JSON_OBJECT(biz_info,"$.sub_vip_type") AS 飞跃会员类型分组
	       ,risk_price                                 AS 风险原始定价
		   ,vip_classify_group
	FROM xyf_dwd.dwd_event_tracking_log_di
	WHERE pt >= '20260123'  
	AND date(TO_DATE(pt, "yyyyMMdd")) = date(trigger_timestamp)
	AND tracking_id = 'XYF_H5_003082' --借款页曝光、走什么会员卡决策流、什么定价都已经决定好了
	AND user_no NOT IN ("1061112123", "1055063706" , "1043199921" , "1028160229" , "1034141205" ) 
	-- AND vip_classify_group = 'fy_group_2' --仅筛选飞跃转化流可见用户 
    QUALIFY ROW_NUMBER() OVER(PARTITION BY user_no , date(trigger_timestamp) ORDER BY trigger_timestamp ASC ) = 1 ---每日首曝口径 
) m 
LEFT JOIN
(
	SELECT  cust_no
		   ,app_user_id
	FROM xyf_dim.dim_user_app_basic_info_df
	WHERE pt = MAX_PT('xyf_dim.dim_user_app_basic_info_df')
	AND app IN ('xyf01','fxk')
) c  --打上cust_no
ON m.user_no = c.app_user_id
LEFT JOIN
(
	SELECT  user_no
	       ,create_date
	       ,MAX(CASE WHEN riskPrice = 'I20' THEN 1 ELSE 0 END) AS risk_I20 --有的话归为1，没有归为0
	FROM
	(
		SELECT  date(create_date) AS create_date
		       ,create_time
		       ,user_no
		       ,riskprice
		FROM xyf_dwd.dwd_lendtrade_rout_info_df --新的中间表 
		WHERE pt = MAX_PT('xyf_dwd.dwd_lendtrade_rout_info_df')
	)
	GROUP BY  user_no
	         ,create_date
)zijin
ON m.user_no = zijin.user_no AND m.pt = zijin.create_date
--消金20 1/22灰度稳定 
LEFT JOIN
(
	SELECT  user_no
	       ,date(trigger_timestamp)                            AS pt
	       ,trigger_timestamp                                  AS 埋点时间
	       ,GET_JSON_OBJECT(biz_info,"$.userAvailableAmt")/100 AS debit_amt --可用额度 
	FROM xyf_dwd.dwd_event_tracking_log_di
	WHERE pt >= '20260122'
	AND date(TO_DATE(pt, "yyyyMMdd")) = date(trigger_timestamp)
	AND tracking_id in('XYF_H5_003241') --价格测试埋点 
	AND user_no NOT IN ("1061112123", "1055063706" , "1043199921" , "1028160229" , "1034141205" ) 
    QUALIFY ROW_NUMBER() OVER(PARTITION BY user_no, date(trigger_timestamp) ORDER BY trigger_timestamp ASC ) = 1 ---取当天第一条 
)price_test
ON m.user_no = price_test.user_no AND m.pt = price_test.pt
-- 飞跃签约表
LEFT JOIN
(
	SELECT  order_time
		   ,vip_order_number
		   ,vip_flow_group
	       ,app_user_id
	       ,date(pay_time)            AS 支付日期
	       ,loan_status               AS fy_loan_status
		   ,pay_time                  AS fy_pay_time
	       ,real_card_price           AS fy_real_card_price
		   ,act_refund_time           AS fy_act_refund_time
		   ,refund_amount             AS fy_refund_amt
	       ,asset_type                AS fy_asset_type
	       ,loan_amt                  AS fy_loan_amt
	       ,sub_vip_type              AS fy_sub_vip_type
	       ,order_amt                 AS fy_order_amt
	FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
	WHERE pt = MAX_PT ('xyf_dwd.dwd_inloan_leap_vip_order_hf')
	AND substr(order_time, 1, 10) >= '2026-01-22'
	AND app_user_id NOT IN ("1061112123", "1055063706", "1043199921", "1028160229", "1034141205")
	AND order_time IS NOT NULL
	-- AND first_vip_order_number = vip_order_number  
	QUALIFY ROW_NUMBER() OVER(PARTITION BY app_user_id, DATE(order_time) ORDER BY order_time DESC) = 1 --取当天最后一条主动签约 
) feiyue_qianyue
ON m.user_no = feiyue_qianyue.app_user_id AND m.pt = DATE(feiyue_qianyue.order_time)
-- 能跟订单一一对应，但存在曝光日多条申请记录
LEFT JOIN
(
	SELECT  order_number
	       ,user_no
	       ,cust_no
	       ,first_order_number
	       ,first_order_time
	       ,order_amt
	       ,risk_status
	       ,loan_status
	       ,loan_time
	       ,loan_amt
	       ,period
	       ,asset_type_flag
	       ,fee_rate
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
	AND app IN ('xyf01')
	AND business_line IN ('APP', '小程序端')
	AND DATE(first_order_time) >= '2026-01-22'
	AND loan_flag <> '首贷' 
) lo
ON m.user_no = lo.user_no AND DATE(lo.first_order_time) = m.pt
-- 关联首贷用户信息
LEFT JOIN
(
	SELECT  user_no
	       ,cust_no
	       ,DATE(loan_time)           AS 首贷时间
	       ,loan_amt
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
	AND app IN ('xyf01','fxk')
	AND loan_status = 'success'
	AND loan_flag = '首贷' 
)shoudai
ON shoudai.cust_no = c.cust_no  AND m.pt > shoudai.首贷时间
WHERE shoudai.user_no IS NOT NULL --筛选复贷用户


In [ ]:
-- by月曝光首笔
SELECT  曝光月
       ,风险原始定价
       ,vip_classify_group
       ,飞跃会员类型分组
       ,group_tag
       ,risk_I20
       ,额度区间
       ,可用额度4k_10k
       ,COUNT(DISTINCT user_no)                                                                                                     AS 曝光人数
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) = 0 THEN user_no END)                                     AS 提现人数_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) BETWEEN 0 AND 3 THEN user_no END)                         AS 提现人数_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) BETWEEN 0 AND 7 THEN user_no END)                         AS 提现人数_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) >= 0 THEN user_no END)                                    AS 提现人数_t30
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN user_no END)                                            AS 放款人数_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN user_no END)                                AS 放款人数_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN user_no END)                                AS 放款人数_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN user_no END)                                           AS 放款人数_t30
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN loan_amt ELSE 0 END)                                               AS 放款金额_t0
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN loan_amt ELSE 0 END)                                   AS 放款金额_t3
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN loan_amt ELSE 0 END)                                   AS 放款金额_t7
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN loan_amt ELSE 0 END)                                              AS 放款金额_t30
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN loan_amt * period ELSE 0 END)                                      AS 期限_t0
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN loan_amt * period ELSE 0 END)                          AS 期限_t3
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN loan_amt * period ELSE 0 END)                          AS 期限_t7
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN loan_amt * period ELSE 0 END)                                     AS 期限_t30
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN loan_amt * fee_rate ELSE 0 END)                                    AS 定价_t0
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN loan_amt * fee_rate ELSE 0 END)                        AS 定价_t3
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN loan_amt * fee_rate ELSE 0 END)                        AS 定价_t7
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN loan_amt * fee_rate ELSE 0 END)                                   AS 定价_t30
       ,SUM(CASE WHEN asset_type_flag = 'I20' AND DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN loan_amt ELSE 0 END)                   AS 放款20金额_t0
       ,SUM(CASE WHEN asset_type_flag = 'I20' AND DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN loan_amt ELSE 0 END)       AS 放款20金额_t3
       ,SUM(CASE WHEN asset_type_flag = 'I20' AND DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN loan_amt ELSE 0 END)       AS 放款20金额_t7
       ,SUM(CASE WHEN asset_type_flag = 'I20' AND DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN loan_amt ELSE 0 END)                  AS 放款20金额_t30
--提现次数 
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) = 0 THEN order_number END)                                AS 提现次数_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) BETWEEN 0 AND 3 THEN order_number END)                    AS 提现次数_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) BETWEEN 0 AND 7 THEN order_number END)                    AS 提现次数_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) >= 0 THEN order_number END)                               AS 提现次数_t30
--风险通过率 
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(风险通过时间) - UNIX_TIMESTAMP(first_order_time)) <= 2 * 3600 THEN order_number END)  AS 风险通过_2h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(风险通过时间) - UNIX_TIMESTAMP(first_order_time)) <= 24 * 3600 THEN order_number END) AS 风险通过_24h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(风险通过时间) - UNIX_TIMESTAMP(first_order_time)) <= 72 * 3600 THEN order_number END) AS 风险通过_72h
       ,COUNT(DISTINCT CASE WHEN 风险通过时间 IS NOT NULL THEN order_number END)                                                       AS 风险通过
--资金通过率 
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(loan_time) - UNIX_TIMESTAMP(风险通过时间)) <= 2 * 3600 THEN order_number END)         AS 资金通过_2h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(loan_time) - UNIX_TIMESTAMP(风险通过时间)) <= 24 * 3600 THEN order_number END)        AS 资金通过_24h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(loan_time) - UNIX_TIMESTAMP(风险通过时间)) <= 72 * 3600 THEN order_number END)        AS 资金通过_72h
       ,COUNT(DISTINCT CASE WHEN loan_time IS NOT NULL THEN order_number END)                                                         AS 资金通过
--放款通过 
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN order_number END)                                       AS 放款通过_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN order_number END)                           AS 放款通过_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN order_number END)                           AS 放款通过_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN order_number END)                                      AS 放款通过_t30
FROM
(
	SELECT  m.曝光月
              ,m.user_no
              ,m.曝光日期
	       ,m.风险原始定价
	       ,m.vip_classify_group
	       ,m.飞跃会员类型分组
	       ,m.group_tag
	       ,m.risk_I20
	       ,m.额度区间
	       ,m.可用额度4k_10k
	       ,lo.*except(user_no)
	       ,b.风险通过时间
	FROM
	(
		SELECT  *
		       ,SUBSTR(曝光日期,1,7) AS 曝光月
		FROM xyf_jingying_dev.price20_preroute_cust_lss5
		WHERE 1 = 1
		AND 曝光日期 >= '2026-01-23'
		AND 曝光日期 <= '2026-05-08' --切全量之前 
              QUALIFY ROW_NUMBER () OVER ( PARTITION BY user_no, SUBSTR(曝光日期, 1, 7) ORDER BY 曝光日期 ASC ) = 1 
	) m
	LEFT JOIN
	(
		SELECT  order_number
		       ,user_no
		       ,cust_no
		       ,first_order_number
		       ,first_order_time
		       ,order_amt
		       ,risk_status
		       ,loan_status
		       ,loan_time
		       ,loan_amt
		       ,period
		       ,asset_type_flag
		       ,fee_rate
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
		AND app IN ('xyf01')
		AND business_line IN ('APP', '小程序端')
		AND DATE(first_order_time) >= '2026-01-22'
		AND loan_flag <> '首贷' 
	) lo
	ON m.user_no = lo.user_no AND DATEDIFF(DATE(lo.first_order_time), m.曝光日期) BETWEEN 0 AND 30
	LEFT JOIN xyf_jingying.weekly_analysis_report_df_lss b
	ON lo.order_number = b.order_number
)
GROUP BY  曝光月
         ,风险原始定价
         ,vip_classify_group
         ,飞跃会员类型分组
         ,group_tag
         ,risk_I20
         ,额度区间
         ,可用额度4k_10k

#### 关联签约当笔

In [1]:
query ='''
-- 关联签约当笔
WITH loan_order_detail AS
(
	SELECT  m.曝光月
	       ,m.user_no
	       ,m.曝光日期
	       ,m.风险原始定价
	       ,m.vip_classify_group
	       ,m.飞跃会员类型分组
	       ,m.group_tag
	       ,m.risk_I20
	       ,m.额度区间
	       ,m.可用额度4k_10k
		   ,DATEDIFF(DATE(lo.first_order_time), m.曝光日期) AS 发起距首曝天数
	       ,lo.*except(user_no)
	FROM
	(
		SELECT  *
		       ,SUBSTR(曝光日期,1,7) AS 曝光月
		FROM xyf_jingying_dev.price20_preroute_cust_lss5
		WHERE 1 = 1
		AND 曝光日期 >= '2026-01-23'
		AND 曝光日期 <= '2026-05-08' --切全量之前 
 		QUALIFY ROW_NUMBER () OVER ( PARTITION BY user_no, SUBSTR(曝光日期, 1, 7) ORDER BY 曝光日期 ASC ) = 1 
	) m
	LEFT JOIN
	(
		SELECT  order_number
		       ,user_no
		       ,first_order_number
		       ,first_order_time
		       ,order_amt
		       ,risk_status
		       ,loan_status
		       ,loan_time
		       ,loan_amt
		       ,period
		       ,asset_type_flag
		       ,fee_rate
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
		AND app IN ('xyf01')
		AND business_line IN ('APP', '小程序端')
		AND DATE(first_order_time) >= '2026-01-22'
		AND loan_flag <> '首贷'
		AND loan_status = 'success' 
	) lo
	ON m.user_no = lo.user_no AND DATEDIFF(DATE(lo.first_order_time), m.曝光日期) BETWEEN 0 AND 30
), repay_plan AS
(
	SELECT  order_number
	       ,SUM(initial_principal)                                                          AS principal
	       ,SUM(initial_interest) + SUM(initial_after_loan_fee) + SUM(initial_platform_fee) AS interest_fee
	FROM xyf_dwd.dwd_repay_loan_repay_plan_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_repay_loan_repay_plan_df')
	GROUP BY  order_number
),

 vip_raw AS
(--是否在提现页买卡？ 
	SELECT  loan_order_number 
	       --, order_status AS status
	       ,real_card_price
	       ,order_time 
		   --, CASE WHEN order_from IN ('lend-before', "lend_before_retain") THEN '借款页签约' END AS is_借款页签约
	       ,pay_time
	       ,act_refund_time
	       ,refund_amount
	       ,'飞享'                                                                                                     AS card_type
	FROM xyf_dwd.dwd_inloan_vip_order_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_vip_order_df')
	AND vip_order_number = first_vip_order_number --不看续约 
	AND order_time IS NOT NULL 

	UNION ALL
	SELECT  loan_order_number 
	       --, CASE WHEN order_status = 'pay_success' THEN 3 ELSE 0 END AS status
	       ,real_card_price
	       ,order_time 
		   --, CASE WHEN order_from LIKE '%lend_before%' THEN '借款页签约' WHEN order_from LIKE '%lend_after%' THEN '卡单页签约' ELSE '其他' END AS is_借款页签约
	       ,pay_time
	       ,act_refund_time
	       ,refund_amount
	       ,'飞跃'                                                                                                   AS card_type
	FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
	WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
	AND vip_order_number = first_vip_order_number --不看续约 
	AND order_time IS NOT NULL

	UNION ALL
	SELECT  loan_order_number
	       ,real_order_price AS real_card_price
	       ,order_time
	       ,order_time       AS pay_time
	       ,act_refund_time
	       ,refund_amount
	       ,'提额'            AS card_type
	FROM xyf_dwd.dwd_user_tek_order_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_user_tek_order_df') 
), vip_by_loan AS
(
	SELECT  loan_order_number
	       ,SUM(CASE WHEN card_type = '飞享' THEN real_card_price ELSE 0 END) AS 飞享签约金额
	       ,SUM(CASE WHEN card_type = '飞跃' THEN real_card_price ELSE 0 END) AS 飞跃签约金额
	       ,SUM(CASE WHEN card_type = '提额' THEN real_card_price ELSE 0 END) AS 提额签约金额
	       ,SUM(CASE WHEN card_type = '飞享' AND pay_time IS NOT NULL THEN real_card_price ELSE 0 END) 
		       - SUM(CASE WHEN card_type = '飞享' AND act_refund_time IS NOT NULL THEN NVL(refund_amount,0) ELSE 0 END) AS 飞享权益收入
	       ,SUM(CASE WHEN card_type = '飞跃' AND pay_time IS NOT NULL THEN real_card_price ELSE 0 END) 
		       - SUM(CASE WHEN card_type = '飞跃' AND act_refund_time IS NOT NULL THEN NVL(refund_amount,0) ELSE 0 END) AS 飞跃权益收入
	       ,SUM(CASE WHEN card_type = '提额' AND pay_time IS NOT NULL THEN real_card_price ELSE 0 END) 
		       - SUM(CASE WHEN card_type = '提额' AND act_refund_time IS NOT NULL THEN NVL(refund_amount,0) ELSE 0 END) AS 提额权益收入
	FROM vip_raw
	GROUP BY  loan_order_number
) , order_level AS
(
	SELECT  a.*
	       ,b.principal
	       ,b.interest_fee
	       ,NVL(v.飞享签约金额,0) AS 飞享签约金额
	       ,NVL(v.飞跃签约金额,0) AS 飞跃签约金额
	       ,NVL(v.提额签约金额,0) AS 提额签约金额
	       ,NVL(v.飞享权益收入,0) AS 飞享权益收入
	       ,NVL(v.飞跃权益收入,0) AS 飞跃权益收入
	       ,NVL(v.提额权益收入,0) AS 提额权益收入
	FROM loan_order_detail a
	LEFT JOIN repay_plan b
	ON a.order_number = b.order_number
	LEFT JOIN vip_by_loan v
	ON a.first_order_number = v.loan_order_number
)


SELECT  曝光月
       ,风险原始定价
       ,vip_classify_group
       ,飞跃会员类型分组
       ,group_tag
       ,risk_I20
       ,额度区间
       ,可用额度4k_10k
       ,COUNT(DISTINCT user_no) AS 首曝人数
       -- T0 累计
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN loan_amt ELSE 0 END) AS 放款金额_T0
       ,COUNT(DISTINCT CASE WHEN 发起距首曝天数 = 0 THEN order_number END) AS 放款笔数_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN loan_amt * fee_rate ELSE 0 END) AS 定价_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN loan_amt * period ELSE 0 END) AS 期限_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN principal ELSE 0 END) AS principal_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN interest_fee ELSE 0 END) AS annualized_interest_fee_T0
	   ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN 飞享签约金额 ELSE 0 END) AS 飞享签约金额_T0
	   ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN 飞跃签约金额 ELSE 0 END) AS 飞跃签约金额_T0
	   ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN 提额签约金额 ELSE 0 END) AS 提额签约金额_T0
	   ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN 飞享权益收入 ELSE 0 END) AS 飞享权益收入_T0
	   ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN 飞跃权益收入 ELSE 0 END) AS 飞跃权益收入_T0
	   ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN 提额权益收入 ELSE 0 END) AS 提额权益收入_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN 飞享权益收入 + 飞跃权益收入 + 提额权益收入 ELSE 0 END) AS 总权益收入_T0

       -- T7 累计：包含 T0 到 T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN loan_amt ELSE 0 END) AS 放款金额_T7
       ,COUNT(DISTINCT CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN order_number END) AS 放款笔数_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN loan_amt * fee_rate ELSE 0 END) AS 定价_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN loan_amt * period ELSE 0 END) AS 期限_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN principal ELSE 0 END) AS principal_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN interest_fee ELSE 0 END) AS annualized_interest_fee_T7
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN 飞享签约金额 ELSE 0 END) AS 飞享签约金额_T7
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN 飞跃签约金额 ELSE 0 END) AS 飞跃签约金额_T7
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN 提额签约金额 ELSE 0 END) AS 提额签约金额_T7
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN 飞享权益收入 ELSE 0 END) AS 飞享权益收入_T7
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN 飞跃权益收入 ELSE 0 END) AS 飞跃权益收入_T7
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN 提额权益收入 ELSE 0 END) AS 提额权益收入_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN 飞享权益收入 + 飞跃权益收入 + 提额权益收入 ELSE 0 END) AS 总权益收入_T7

       -- T30 累计：包含 T0 到 T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN loan_amt ELSE 0 END) AS 放款金额_T30
       ,COUNT(DISTINCT CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN order_number END) AS 放款笔数_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN loan_amt * fee_rate ELSE 0 END) AS 定价_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN loan_amt * period ELSE 0 END) AS 期限_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN principal ELSE 0 END) AS principal_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN interest_fee ELSE 0 END) AS annualized_interest_fee_T30
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN 飞享签约金额 ELSE 0 END) AS 飞享签约金额_T30
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN 飞跃签约金额 ELSE 0 END) AS 飞跃签约金额_T30
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN 提额签约金额 ELSE 0 END) AS 提额签约金额_T30
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN 飞享权益收入 ELSE 0 END) AS 飞享权益收入_T30
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN 飞跃权益收入 ELSE 0 END) AS 飞跃权益收入_T30
	   ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN 提额权益收入 ELSE 0 END) AS 提额权益收入_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN 飞享权益收入 + 飞跃权益收入 + 提额权益收入 ELSE 0 END) AS 总权益收入_T30
FROM order_level
GROUP BY  曝光月
         ,风险原始定价
         ,vip_classify_group
         ,飞跃会员类型分组
         ,group_tag
         ,risk_I20
         ,额度区间
         ,可用额度4k_10k;
'''

In [2]:
import query_analysis_tool as qat
loan_stats = qat.run_query(query)

# 字符串字段
str_cols = ['曝光月','风险原始定价','vip_classify_group','飞跃会员类型分组','group_tag','risk_I20','额度区间','可用额度4k_10k']


# 浮点数字段 - 保留2位小数，所有有应还本金的字段均为浮点数

float_cols = [ '放款金额_T0', '放款金额_T7', '放款金额_T30',
                '定价_T0', '定价_T7', '定价_T30',
                '期限_T0', '期限_T7', '期限_T30',
                'principal_T0', 'principal_T7', 'principal_T30',
                'annualized_interest_fee_T0', 'annualized_interest_fee_T7', 'annualized_interest_fee_T30',
                '飞享签约金额_T0', '飞享签约金额_T7', '飞享签约金额_T30',
                '飞跃签约金额_T0', '飞跃签约金额_T7', '飞跃签约金额_T30',
                '提额签约金额_T0', '提额签约金额_T7', '提额签约金额_T30',
                '飞享权益收入_T0', '飞享权益收入_T7', '飞享权益收入_T30',
                '飞跃权益收入_T0', '飞跃权益收入_T7', '飞跃权益收入_T30',
                '提额权益收入_T0', '提额权益收入_T7', '提额权益收入_T30',
                '总权益收入_T0', '总权益收入_T7', '总权益收入_T30']

# 整数字段
int_cols = [ '首曝人数', '放款笔数_T0', '放款笔数_T7', '放款笔数_T30']

loan_stats = qat.format_dataframe_columns(
    loan_stats,
    str_cols=str_cols,
    date_cols=[], 
    int_cols=int_cols,
    float_cols=float_cols
)

qat.write_dataframe_to_excel(
    file_path=r"D:\4.临时取数\20+权益资产测试\消金20接资金预路由老客0528_full.xlsx",
    dataframes_dict={"会员卡细分收入": loan_stats},
    start_row=1,
    include_header=True
)

正在获取数据，首段 SQL: 
-- 关联签约当笔
WITH loan_order_detail AS
(
	SELECT  m. ...
成功写入工作表: 会员卡细分收入
文件已保存: D:\4.临时取数\20+权益资产测试\消金20接资金预路由老客0528_full.xlsx


In [6]:
import query_analysis_tool as qat


calc_fields = {
    "T0放款件均": {"formula": "='放款金额_T0'/'放款笔数_T0'", "number_format": "0.00%"},
    "T7放款件均": {"formula": "='放款金额_T7'/'放款笔数_T7'", "number_format": "0.00%"},
    "T30放款件均": {"formula": "='放款金额_T30'/'放款笔数_T30'", "number_format": "0.00%"},

    "T0加权期限": {"formula": "='期限_T0'/'放款金额_T0'", "number_format": "0.00"},
    "T7加权期限": {"formula": "='期限_T7'/'放款金额_T7'", "number_format": "0.00"},
    "T30加权期限": {"formula": "='期限_T30'/'放款金额_T30'", "number_format": "0.00"},

    "T0加权定价": {"formula": "='定价_T0'/'放款金额_T0'", "number_format": "0.00%"},
    "T7加权定价": {"formula": "='定价_T7'/'放款金额_T7'", "number_format": "0.00%"},
    "T30加权定价": {"formula": "='定价_T30'/'放款金额_T30'", "number_format": "0.00%"},

    "T0息费率": {"formula": "='annualized_interest_fee_T0'/'principal_T0'", "number_format": "0.00%"},
    "T7息费率": {"formula": "='annualized_interest_fee_T7'/'principal_T7'", "number_format": "0.00%"},
    "T30息费率": {"formula": "='annualized_interest_fee_T30'/'principal_T30'", "number_format": "0.00%"},

    "T0权益收入占放款": {"formula": "='总权益收入_T0'/'放款金额_T0'", "number_format": "0.00%"},
    "T7权益收入占放款": {"formula": "='总权益收入_T7'/'放款金额_T7'", "number_format": "0.00%"},
    "T30权益收入占放款": {"formula": "='总权益收入_T30'/'放款金额_T30'", "number_format": "0.00%"},

    "T0飞享签约金额": {"formula": "='飞享签约金额_T0'/'放款金额_T0'", "number_format": "0.00%"},
    "T7飞享签约金额": {"formula": "='飞享签约金额_T7'/'放款金额_T7'", "number_format": "0.00%"},
    "T30飞享签约金额": {"formula": "='飞享签约金额_T30'/'放款金额_T30'", "number_format": "0.00%"},
    "T0飞享权益收入": {"formula": "='飞享权益收入_T0'/'放款金额_T0'", "number_format": "0.00%"},
    "T7飞享权益收入": {"formula": "='飞享权益收入_T7'/'放款金额_T7'", "number_format": "0.00%"},
    "T30飞享权益收入": {"formula": "='飞享权益收入_T30'/'放款金额_T30'", "number_format": "0.00%"},

    "T0飞跃签约金额": {"formula": "='飞跃签约金额_T0'/'放款金额_T0'", "number_format": "0.00%"},
    "T7飞跃签约金额": {"formula": "='飞跃签约金额_T7'/'放款金额_T7'", "number_format": "0.00%"},
    "T30飞跃签约金额": {"formula": "='飞跃签约金额_T30'/'放款金额_T30'", "number_format": "0.00%"},
    "T0飞跃权益收入": {"formula": "='飞跃权益收入_T0'/'放款金额_T0'", "number_format": "0.00%"},
    "T7飞跃权益收入": {"formula": "='飞跃权益收入_T7'/'放款金额_T7'", "number_format": "0.00%"},
    "T30飞跃权益收入": {"formula": "='飞跃权益收入_T30'/'放款金额_T30'", "number_format": "0.00%"},

    "T0提额卡签约金额": {"formula": "='提额签约金额_T0'/'放款金额_T0'", "number_format": "0.00%"},
    "T7提额卡签约金额": {"formula": "='提额签约金额_T7'/'放款金额_T7'", "number_format": "0.00%"},
    "T30提额卡签约金额": {"formula": "='提额签约金额_T30'/'放款金额_T30'", "number_format": "0.00%"},
    "T0提额卡权益收入": {"formula": "='提额权益收入_T0'/'放款金额_T0'", "number_format": "0.00%"},
    "T7提额卡权益收入": {"formula": "='提额权益收入_T7'/'放款金额_T7'", "number_format": "0.00%"},
    "T30提额卡权益收入": {"formula": "='提额权益收入_T30'/'放款金额_T30'", "number_format": "0.00%"},

}

qat.add_pivot_calculated_fields(
    file_path=r"D:\4.临时取数\20+权益资产测试\消金20接资金预路由老客0528_full.xlsx",
    sheet_name="summary",
    pivot_name="数据透视表15",
    fields=calc_fields
) 


[Pivot] sheet=summary pivot=数据透视表15 cache_index=7
[Skip] T0放款件均 已存在，跳过
[Skip] T7放款件均 已存在，跳过
[Skip] T30放款件均 已存在，跳过
[Skip] T0加权期限 已存在，跳过
[Skip] T7加权期限 已存在，跳过
[Skip] T30加权期限 已存在，跳过
[Skip] T0加权定价 已存在，跳过
[Skip] T7加权定价 已存在，跳过
[Skip] T30加权定价 已存在，跳过
[Skip] T0息费率 已存在，跳过
[Skip] T7息费率 已存在，跳过
[Skip] T30息费率 已存在，跳过
[Skip] T0权益收入占放款 已存在，跳过
[Skip] T7权益收入占放款 已存在，跳过
[Skip] T30权益收入占放款 已存在，跳过
[Skip] T0飞享签约金额 已存在，跳过
[Skip] T7飞享签约金额 已存在，跳过
[Skip] T30飞享签约金额 已存在，跳过
[Skip] T0飞享权益收入 已存在，跳过
[Skip] T7飞享权益收入 已存在，跳过
[Skip] T30飞享权益收入 已存在，跳过
[Skip] T0飞跃签约金额 已存在，跳过
[Skip] T7飞跃签约金额 已存在，跳过
[Skip] T30飞跃签约金额 已存在，跳过
[Skip] T0飞跃权益收入 已存在，跳过
[Skip] T7飞跃权益收入 已存在，跳过
[Skip] T30飞跃权益收入 已存在，跳过
[OK] Add CalculatedField: T0提额卡签约金额 | ='提额签约金额_T0'/'放款金额_T0'
[OK] Add to Values: T0提额卡签约金额 | format=0.00%
[OK] Add CalculatedField: T7提额卡签约金额 | ='提额签约金额_T7'/'放款金额_T7'
[OK] Add to Values: T7提额卡签约金额 | format=0.00%
[OK] Add CalculatedField: T30提额卡签约金额 | ='提额签约金额_T30'/'放款金额_T30'
[OK] Add to Values: T30提额卡签约金额 | format=0.00%
[OK] Add CalculatedFie

[]

#### 发起、风险通过率

In [ ]:
query = '''
-- by月曝光首笔
SELECT  曝光月
       ,风险原始定价
       ,vip_classify_group
       ,飞跃会员类型分组
       ,group_tag
       ,risk_I20
       ,额度区间
       ,可用额度4k_10k
       ,COUNT(DISTINCT user_no)                                                                                                     AS 曝光人数
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) = 0 THEN user_no END)                                     AS 提现人数_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) BETWEEN 0 AND 3 THEN user_no END)                         AS 提现人数_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) BETWEEN 0 AND 7 THEN user_no END)                         AS 提现人数_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) >= 0 THEN user_no END)                                    AS 提现人数_t30
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN user_no END)                                            AS 放款人数_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN user_no END)                                AS 放款人数_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN user_no END)                                AS 放款人数_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN user_no END)                                           AS 放款人数_t30
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN loan_amt ELSE 0 END)                                               AS 放款金额_t0
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN loan_amt ELSE 0 END)                                   AS 放款金额_t3
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN loan_amt ELSE 0 END)                                   AS 放款金额_t7
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN loan_amt ELSE 0 END)                                              AS 放款金额_t30
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN loan_amt * period ELSE 0 END)                                      AS 期限_t0
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN loan_amt * period ELSE 0 END)                          AS 期限_t3
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN loan_amt * period ELSE 0 END)                          AS 期限_t7
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN loan_amt * period ELSE 0 END)                                     AS 期限_t30
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN loan_amt * fee_rate ELSE 0 END)                                    AS 定价_t0
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN loan_amt * fee_rate ELSE 0 END)                        AS 定价_t3
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN loan_amt * fee_rate ELSE 0 END)                        AS 定价_t7
       ,SUM(CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN loan_amt * fee_rate ELSE 0 END)                                   AS 定价_t30
       ,SUM(CASE WHEN asset_type_flag = 'I20' AND DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN loan_amt ELSE 0 END)                   AS 放款20金额_t0
       ,SUM(CASE WHEN asset_type_flag = 'I20' AND DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN loan_amt ELSE 0 END)       AS 放款20金额_t3
       ,SUM(CASE WHEN asset_type_flag = 'I20' AND DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN loan_amt ELSE 0 END)       AS 放款20金额_t7
       ,SUM(CASE WHEN asset_type_flag = 'I20' AND DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN loan_amt ELSE 0 END)                  AS 放款20金额_t30
--提现次数 
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) = 0 THEN order_number END)                                AS 提现次数_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) BETWEEN 0 AND 3 THEN order_number END)                    AS 提现次数_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) BETWEEN 0 AND 7 THEN order_number END)                    AS 提现次数_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(first_order_time),曝光日期) >= 0 THEN order_number END)                               AS 提现次数_t30
--风险通过率 
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(风险通过时间) - UNIX_TIMESTAMP(first_order_time)) <= 2 * 3600 THEN order_number END)  AS 风险通过_2h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(风险通过时间) - UNIX_TIMESTAMP(first_order_time)) <= 24 * 3600 THEN order_number END) AS 风险通过_24h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(风险通过时间) - UNIX_TIMESTAMP(first_order_time)) <= 72 * 3600 THEN order_number END) AS 风险通过_72h
       ,COUNT(DISTINCT CASE WHEN 风险通过时间 IS NOT NULL THEN order_number END)                                                       AS 风险通过
--资金通过率 
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(loan_time) - UNIX_TIMESTAMP(风险通过时间)) <= 2 * 3600 THEN order_number END)         AS 资金通过_2h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(loan_time) - UNIX_TIMESTAMP(风险通过时间)) <= 24 * 3600 THEN order_number END)        AS 资金通过_24h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(loan_time) - UNIX_TIMESTAMP(风险通过时间)) <= 72 * 3600 THEN order_number END)        AS 资金通过_72h
       ,COUNT(DISTINCT CASE WHEN loan_time IS NOT NULL THEN order_number END)                                                         AS 资金通过
--放款通过 
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) = 0 THEN order_number END)                                       AS 放款通过_t0
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 3 THEN order_number END)                           AS 放款通过_t3
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) BETWEEN 0 AND 7 THEN order_number END)                           AS 放款通过_t7
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(DATE(loan_time),曝光日期) >= 0 THEN order_number END)                                      AS 放款通过_t30
FROM
(
	SELECT  m.曝光月
              ,m.user_no
              ,m.曝光日期
	       ,m.风险原始定价
	       ,m.vip_classify_group
	       ,m.飞跃会员类型分组
	       ,m.group_tag
	       ,m.risk_I20
	       ,m.额度区间
	       ,m.可用额度4k_10k
	       ,lo.*except(user_no)
	       ,b.风险通过时间
	FROM
	(
		SELECT  *
		       ,SUBSTR(曝光日期,1,7) AS 曝光月
		FROM xyf_jingying_dev.price20_preroute_cust_lss5
		WHERE 1 = 1
		AND 曝光日期 >= '2026-01-23'
		AND 曝光日期 <= '2026-05-08' --切全量之前 
              QUALIFY ROW_NUMBER () OVER ( PARTITION BY user_no, SUBSTR(曝光日期, 1, 7) ORDER BY 曝光日期 ASC ) = 1 
	) m
	LEFT JOIN
	(
		SELECT  order_number
		       ,user_no
		       ,cust_no
		       ,first_order_number
		       ,first_order_time
		       ,order_amt
		       ,risk_status
		       ,loan_status
		       ,loan_time
		       ,loan_amt
		       ,period
		       ,asset_type_flag
		       ,fee_rate
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
		AND app IN ('xyf01')
		AND business_line IN ('APP', '小程序端')
		AND DATE(first_order_time) >= '2026-01-22'
		AND loan_flag <> '首贷' 
	) lo
	ON m.user_no = lo.user_no AND DATEDIFF(DATE(lo.first_order_time), m.曝光日期) BETWEEN 0 AND 30
	LEFT JOIN xyf_jingying.weekly_analysis_report_df_lss b
	ON lo.order_number = b.order_number
)
GROUP BY  曝光月
         ,风险原始定价
         ,vip_classify_group
         ,飞跃会员类型分组
         ,group_tag
         ,risk_I20
         ,额度区间
         ,可用额度4k_10k
'''

In [6]:
import query_analysis_tool as qat
loan_stats = qat.run_query(query)

# 字符串字段
str_cols = ['曝光月','风险原始定价','vip_classify_group','飞跃会员类型分组','group_tag','risk_I20','额度区间','可用额度4k_10k']


# 浮点数字段 - 保留2位小数，所有有应还本金的字段均为浮点数

float_cols = [ '放款金额_t0','放款金额_t3','放款金额_t7','放款金额_t30',
                '期限_t0','期限_t3','期限_t7','期限_t30',
                '定价_t0','定价_t3','定价_t7','定价_t30',
                 '放款20金额_t0','放款20金额_t3','放款20金额_t7','放款20金额_t30'
]

# 整数字段
int_cols = [ '曝光人数','提现人数_t0','提现人数_t3','提现人数_t7','提现人数_t30',
             '提现次数_t0','提现次数_t3','提现次数_t7','提现次数_t30',
             '放款人数_t0','放款人数_t3','放款人数_t7','放款人数_t30',
             '提现次数_t0','提现次数_t3','提现次数_t7','提现次数_t30',
             '风险通过_2h','风险通过_24h','风险通过_72h','风险通过',
             '资金通过_2h','资金通过_24h','资金通过_72h',
             '放款通过_t0','放款通过_t3','放款通过_t7','放款通过_t30'
]

loan_stats = qat.format_dataframe_columns(
    loan_stats,
    str_cols=str_cols,
    date_cols=[], 
    int_cols=int_cols,
    float_cols=float_cols
)

qat.write_dataframe_to_excel(
    file_path=r"D:\4.临时取数\20+权益资产测试\消金20接资金预路由老客0526_full.xlsx",
    dataframes_dict={"月首曝数据": loan_stats},
    start_row=1,
    include_header=True
)


正在获取数据，首段 SQL: 
-- by月曝光首笔
SELECT  曝光月
       ,风险原始定价
       ,vip ...
成功写入工作表: 月首曝数据
文件已保存: D:\4.临时取数\20+权益资产测试\消金20接资金预路由老客0526_full.xlsx


In [12]:
import query_analysis_tool as qat
calc_fields = {
    "T0发起率": {"formula": "='提现人数_t0'/曝光人数", "number_format": "0.00%"},
    "T3发起率": {"formula": "='提现人数_t3'/曝光人数", "number_format": "0.00%"},
    "T7发起率": {"formula": "='提现人数_t7'/曝光人数", "number_format": "0.00%"},
    "T30发起率": {"formula": "='提现人数_t30'/曝光人数", "number_format": "0.00%"},

    "T0放款率": {"formula": "='放款人数_t0'/曝光人数", "number_format": "0.00%"},
    "T3放款率": {"formula": "='放款人数_t3'/曝光人数", "number_format": "0.00%"},
    "T7放款率": {"formula": "='放款人数_t7'/曝光人数", "number_format": "0.00%"},
    "T30放款率": {"formula": "='放款人数_t30'/曝光人数", "number_format": "0.00%"},
    
    'T0期限': {'formula': '=期限_t0/放款金额_t0', 'number_format': '0.00'},
    'T3期限': {'formula': '=期限_t3/放款金额_t3', 'number_format': '0.00'},
    'T7期限': {'formula': '=期限_t7/放款金额_t7', 'number_format': '0.00'},
    'T30期限': {'formula': '=期限_t30/放款金额_t30', 'number_format': '0.00'},

    'T0定价': {'formula': '=定价_t0/放款金额_t0', 'number_format': '0.00'},
    'T3定价': {'formula': '=定价_t3/放款金额_t3', 'number_format': '0.00'},
    'T7定价': {'formula': '=定价_t7/放款金额_t7', 'number_format': '0.00'},
    'T30定价': {'formula': '=定价_t30/放款金额_t30', 'number_format': '0.00'},

    'T0放款20金额': {'formula': '=放款20金额_t0/放款金额_t0', 'number_format': '0.00'},
    'T3放款20金额': {'formula': '=放款20金额_t3/放款金额_t3', 'number_format': '0.00'},
    'T7放款20金额': {'formula': '=放款20金额_t7/放款金额_t7', 'number_format': '0.00'},
    'T30放款20金额': {'formula': '=放款20金额_t30/放款金额_t30', 'number_format': '0.00'},

    "风险2h通过率": {"formula": "='风险通过_2h'/提现次数_t30", "number_format": "0.00%"},
    "风险24h通过率": {"formula": "='风险通过_24h'/提现次数_t30", "number_format": "0.00%"},
    "风险72h通过率": {"formula": "='风险通过_72h'/提现次数_t30", "number_format": "0.00%"},

    "资金2h通过率": {"formula": "='资金通过_2h'/风险通过", "number_format": "0.00%"},
    "资金24h通过率": {"formula": "='资金通过_24h'/风险通过", "number_format": "0.00%"},
    "资金72h通过率": {"formula": "='资金通过_72h'/风险通过", "number_format": "0.00%"},

    "放款次数t0": {"formula": "='放款通过_t0'/提现次数_t0", "number_format": "0.00%"},
    "放款次数t3": {"formula": "='放款通过_t3'/提现次数_t3", "number_format": "0.00%"},
    "放款次数t7": {"formula": "='放款通过_t7'/提现次数_t7", "number_format": "0.00%"},
    "放款次数t30": {"formula": "='放款通过_t30'/提现次数_t30", "number_format": "0.00%"},
}

qat.add_pivot_calculated_fields(
    file_path=r"D:\4.临时取数\20+权益资产测试\消金20接资金预路由老客.xlsx",
    sheet_name="月转化",
    pivot_name="数据透视表2",
    fields=calc_fields
) 


[Pivot] sheet=月转化 pivot=数据透视表2 cache_index=3
[Skip] T0发起率 已存在，跳过
[Skip] T3发起率 已存在，跳过
[Skip] T7发起率 已存在，跳过
[Skip] T30发起率 已存在，跳过
[Skip] T0放款率 已存在，跳过
[Skip] T3放款率 已存在，跳过
[Skip] T7放款率 已存在，跳过
[Skip] T30放款率 已存在，跳过
[Skip] T0期限 已存在，跳过
[Skip] T3期限 已存在，跳过
[Skip] T7期限 已存在，跳过
[Skip] T30期限 已存在，跳过
[Skip] T0定价 已存在，跳过
[Skip] T3定价 已存在，跳过
[Skip] T7定价 已存在，跳过
[Skip] T30定价 已存在，跳过
[Skip] T0放款20金额 已存在，跳过
[Skip] T3放款20金额 已存在，跳过
[Skip] T7放款20金额 已存在，跳过
[Skip] T30放款20金额 已存在，跳过
[Skip] 风险2h通过率 已存在，跳过
[Skip] 风险24h通过率 已存在，跳过
[Skip] 风险72h通过率 已存在，跳过
[Skip] 资金2h通过率 已存在，跳过
[Skip] 资金24h通过率 已存在，跳过
[Skip] 资金72h通过率 已存在，跳过
[OK] Add CalculatedField: 放款次数t0 | ='放款通过_t0'/提现次数_t0
[OK] Add to Values: 放款次数t0 | format=0.00%
[OK] Add CalculatedField: 放款次数t3 | ='放款通过_t3'/提现次数_t3
[OK] Add to Values: 放款次数t3 | format=0.00%
[OK] Add CalculatedField: 放款次数t7 | ='放款通过_t7'/提现次数_t7
[OK] Add to Values: 放款次数t7 | format=0.00%
[OK] Add CalculatedField: 放款次数t30 | ='放款通过_t30'/提现次数_t30
[OK] Add to Values: 放款次数t30 | format=0.00%

全部计算字段设置成功。



[]

#### 会员卡收入（窗口期放款订单与会员卡订单）

In [2]:
query = '''
WITH exposure_base AS
(
	SELECT  *
	       ,SUBSTR(曝光日期,1,7) AS 曝光月
	FROM xyf_jingying_dev.price20_preroute_cust_lss5
	WHERE 1 = 1
	AND 曝光日期 >= '2026-02-01'
	AND 曝光日期 <= '2026-04-30' 
	QUALIFY ROW_NUMBER() OVER ( PARTITION BY user_no, SUBSTR(曝光日期, 1, 7) ORDER BY 曝光日期 ASC ) = 1 
),
-- 放款息费、定价、期限
 loan_order_detail AS
(
	SELECT  m.曝光月
	       ,m.user_no
	       ,m.曝光日期
	       ,m.风险原始定价
	       ,m.vip_classify_group
	       ,m.飞跃会员类型分组
	       ,m.group_tag
	       ,m.risk_I20
	       ,m.额度区间
	       ,m.可用额度4k_10k
	       ,DATEDIFF(DATE(lo.first_order_time),m.曝光日期) AS 发起距首曝天数
	       ,lo.* EXCEPT(user_no)
	FROM exposure_base m
	LEFT JOIN
	(
		SELECT  order_number
		       ,user_no
		       ,first_order_number
		       ,first_order_time
		       ,order_amt
		       ,risk_status
		       ,loan_status
		       ,loan_time
		       ,loan_amt
		       ,period
		       ,asset_type_flag
		       ,fee_rate
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
		AND app IN ('xyf01')
		AND business_line IN ('APP', '小程序端')
		AND DATE(first_order_time) >= '2026-01-22'
		AND loan_flag <> '首贷'
		AND loan_status = 'success' 
	) lo
	ON m.user_no = lo.user_no AND DATEDIFF(DATE(lo.first_order_time), m.曝光日期) BETWEEN 0 AND 90
), repay_plan AS
(
	SELECT  order_number
	       ,SUM(initial_principal)                                                          AS principal
	       ,SUM(initial_interest) + SUM(initial_after_loan_fee) + SUM(initial_platform_fee) AS interest_fee
	FROM xyf_dwd.dwd_repay_loan_repay_plan_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_repay_loan_repay_plan_df')
	GROUP BY  order_number
), order_level AS
(
	SELECT  a.*
	       ,b.principal
	       ,b.interest_fee
	FROM loan_order_detail a
	LEFT JOIN repay_plan b
	ON a.order_number = b.order_number
), 
 loan_agg AS
(
SELECT  曝光月
       --,风险原始定价
       --,vip_classify_group
       --,飞跃会员类型分组
       ,group_tag
       --,risk_I20
       --,额度区间
       ,可用额度4k_10k
       ,COUNT(DISTINCT user_no)                                                           AS 首曝人数

       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN loan_amt ELSE 0 END)                         AS 放款金额_T0
       ,COUNT(DISTINCT CASE WHEN 发起距首曝天数 = 0 THEN order_number END)                 AS 放款笔数_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN loan_amt * fee_rate ELSE 0 END)              AS 定价_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN loan_amt * period ELSE 0 END)                AS 期限_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN principal ELSE 0 END)                        AS principal_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN interest_fee ELSE 0 END)                     AS annualized_interest_fee_T0

       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN loan_amt ELSE 0 END)             AS 放款金额_T7
       ,COUNT(DISTINCT CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN order_number END)     AS 放款笔数_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN loan_amt * fee_rate ELSE 0 END)  AS 定价_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN loan_amt * period ELSE 0 END)    AS 期限_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN principal ELSE 0 END)            AS principal_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN interest_fee ELSE 0 END)         AS annualized_interest_fee_T7

       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN loan_amt ELSE 0 END)            AS 放款金额_T30
       ,COUNT(DISTINCT CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN order_number END)    AS 放款笔数_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN loan_amt * fee_rate ELSE 0 END) AS 定价_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN loan_amt * period ELSE 0 END)   AS 期限_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN principal ELSE 0 END)           AS principal_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN interest_fee ELSE 0 END)        AS annualized_interest_fee_T30

	,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 60 THEN loan_amt ELSE 0 END)            AS 放款金额_T60
       ,COUNT(DISTINCT CASE WHEN 发起距首曝天数 BETWEEN 0 AND 60 THEN order_number END)    AS 放款笔数_T60
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 60 THEN loan_amt * fee_rate ELSE 0 END) AS 定价_T60
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 60 THEN loan_amt * period ELSE 0 END)   AS 期限_T60
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 60 THEN principal ELSE 0 END)           AS principal_T60
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 60 THEN interest_fee ELSE 0 END)        AS annualized_interest_fee_T60

	,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 90 THEN loan_amt ELSE 0 END)            AS 放款金额_T90
       ,COUNT(DISTINCT CASE WHEN 发起距首曝天数 BETWEEN 0 AND 90 THEN order_number END)    AS 放款笔数_T90
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 90 THEN loan_amt * fee_rate ELSE 0 END) AS 定价_T90
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 90 THEN loan_amt * period ELSE 0 END)   AS 期限_T90
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 90 THEN principal ELSE 0 END)           AS principal_T90
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 90 THEN interest_fee ELSE 0 END)        AS annualized_interest_fee_T90
FROM order_level
GROUP BY  曝光月
         --,风险原始定价
         --,vip_classify_group
         --,飞跃会员类型分组
         ,group_tag
         --,risk_I20
         --,额度区间
         ,可用额度4k_10k
),

--会员卡收入
vip_order_raw AS
(
	SELECT  app_user_id
	       ,cust_no
	       ,order_time
	       ,'飞跃'                                                                      AS card_type
	       ,real_card_price                                                             AS sign_amt
	       ,CASE WHEN pay_time IS NOT NULL THEN real_card_price  ELSE 0 END             AS pay_amt
	       ,CASE WHEN act_refund_time IS NOT NULL THEN NVL(refund_amount,0)  ELSE 0 END AS refund_amt
	FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
	WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
	AND DATE(order_time) >= '2025-05-24' 

	UNION ALL
	SELECT  app_user_id
	       ,cust_no
	       ,order_time
	       ,'飞享'                                                                            AS card_type
	       ,real_card_price / 100                                                             AS sign_amt
	       ,CASE WHEN pay_time IS NOT NULL THEN real_card_price / 100  ELSE 0 END             AS pay_amt
	       ,CASE WHEN act_refund_time IS NOT NULL THEN NVL(refund_amount,0) / 100  ELSE 0 END AS refund_amt
	FROM xyf_dwd.dwd_user_vip_order_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_user_vip_order_df')
	AND vip_card_type = 1
	AND if_validation <> 0 
	
	UNION ALL
	SELECT  app_user_id
	       ,cust_no
	       ,order_time
	       ,'提额'                                                                      AS card_type
	       ,real_order_price                                                            AS sign_amt
	       ,real_order_price                                                            AS pay_amt
	       ,CASE WHEN act_refund_time IS NOT NULL THEN NVL(refund_amount,0)  ELSE 0 END AS refund_amt
	FROM xyf_dwd.dwd_user_tek_order_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_user_tek_order_df') 
), vip_order_detail AS
(
	SELECT  m.曝光月
	       ,m.user_no
	       ,m.风险原始定价
	       ,m.vip_classify_group
	       ,m.飞跃会员类型分组
	       ,m.group_tag
	       ,m.risk_I20
	       ,m.额度区间
	       ,m.可用额度4k_10k
	       ,DATEDIFF(DATE(v.order_time),m.曝光日期) AS 会员卡距首曝天数
	       ,v.card_type
	       ,v.sign_amt
	       ,v.pay_amt - v.refund_amt            AS equity_income
	FROM exposure_base m
	LEFT JOIN vip_order_raw v
	ON m.user_no = v.app_user_id 
    AND DATEDIFF(DATE(v.order_time), m.曝光日期) BETWEEN 0 AND 90
), vip_agg AS
(
SELECT  曝光月
       --,风险原始定价
       --,vip_classify_group
       --,飞跃会员类型分组
       ,group_tag
       --,risk_I20
       --,额度区间
       ,可用额度4k_10k
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '飞享' THEN sign_amt ELSE 0 END)                   AS 飞享签约金额_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '飞跃' THEN sign_amt ELSE 0 END)                   AS 飞跃签约金额_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '提额' THEN sign_amt ELSE 0 END)                   AS 提额签约金额_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '飞享' THEN equity_income ELSE 0 END)              AS 飞享权益收入_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '飞跃' THEN equity_income ELSE 0 END)              AS 飞跃权益收入_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '提额' THEN equity_income ELSE 0 END)              AS 提额权益收入_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 THEN equity_income ELSE 0 END)                                     AS 总权益收入_T0

       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '飞享' THEN sign_amt ELSE 0 END)       AS 飞享签约金额_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '飞跃' THEN sign_amt ELSE 0 END)       AS 飞跃签约金额_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '提额' THEN sign_amt ELSE 0 END)       AS 提额签约金额_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '飞享' THEN equity_income ELSE 0 END)  AS 飞享权益收入_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '飞跃' THEN equity_income ELSE 0 END)  AS 飞跃权益收入_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '提额' THEN equity_income ELSE 0 END)  AS 提额权益收入_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 THEN equity_income ELSE 0 END)                         AS 总权益收入_T7

       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '飞享' THEN sign_amt ELSE 0 END)      AS 飞享签约金额_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '飞跃' THEN sign_amt ELSE 0 END)      AS 飞跃签约金额_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '提额' THEN sign_amt ELSE 0 END)      AS 提额签约金额_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '飞享' THEN equity_income ELSE 0 END) AS 飞享权益收入_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '飞跃' THEN equity_income ELSE 0 END) AS 飞跃权益收入_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '提额' THEN equity_income ELSE 0 END) AS 提额权益收入_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 THEN equity_income ELSE 0 END)                        AS 总权益收入_T30

       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 60 AND card_type = '飞享' THEN sign_amt ELSE 0 END)      AS 飞享签约金额_T60
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 60 AND card_type = '飞跃' THEN sign_amt ELSE 0 END)      AS 飞跃签约金额_T60
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 60 AND card_type = '提额' THEN sign_amt ELSE 0 END)      AS 提额签约金额_T60
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 60 AND card_type = '飞享' THEN equity_income ELSE 0 END) AS 飞享权益收入_T60
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 60 AND card_type = '飞跃' THEN equity_income ELSE 0 END) AS 飞跃权益收入_T60
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 60 AND card_type = '提额' THEN equity_income ELSE 0 END) AS 提额权益收入_T60
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 60 THEN equity_income ELSE 0 END)                        AS 总权益收入_T60

       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 90 AND card_type = '飞享' THEN sign_amt ELSE 0 END)      AS 飞享签约金额_T90
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 90 AND card_type = '飞跃' THEN sign_amt ELSE 0 END)      AS 飞跃签约金额_T90
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 90 AND card_type = '提额' THEN sign_amt ELSE 0 END)      AS 提额签约金额_T90
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 90 AND card_type = '飞享' THEN equity_income ELSE 0 END) AS 飞享权益收入_T90
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 90 AND card_type = '飞跃' THEN equity_income ELSE 0 END) AS 飞跃权益收入_T90
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 90 AND card_type = '提额' THEN equity_income ELSE 0 END) AS 提额权益收入_T90
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 90 THEN equity_income ELSE 0 END)                        AS 总权益收入_T90
FROM vip_order_detail
GROUP BY  曝光月
         --,风险原始定价
         --,vip_classify_group
        -- ,飞跃会员类型分组
         ,group_tag
         --,risk_I20
         --,额度区间
         ,可用额度4k_10k
)

SELECT  l.*
       ,NVL(v.飞享签约金额_T0,0)  AS 飞享签约金额_T0
       ,NVL(v.飞跃签约金额_T0,0)  AS 飞跃签约金额_T0
       ,NVL(v.提额签约金额_T0,0)  AS 提额签约金额_T0
       ,NVL(v.飞享权益收入_T0,0)  AS 飞享权益收入_T0
       ,NVL(v.飞跃权益收入_T0,0)  AS 飞跃权益收入_T0
       ,NVL(v.提额权益收入_T0,0)  AS 提额权益收入_T0
       ,NVL(v.总权益收入_T0,0)    AS 总权益收入_T0
       ,NVL(v.飞享签约金额_T7,0)  AS 飞享签约金额_T7
       ,NVL(v.飞跃签约金额_T7,0)  AS 飞跃签约金额_T7
       ,NVL(v.提额签约金额_T7,0)  AS 提额签约金额_T7
       ,NVL(v.飞享权益收入_T7,0)  AS 飞享权益收入_T7
       ,NVL(v.飞跃权益收入_T7,0)  AS 飞跃权益收入_T7
       ,NVL(v.提额权益收入_T7,0)  AS 提额权益收入_T7
       ,NVL(v.总权益收入_T7,0)    AS 总权益收入_T7
       ,NVL(v.飞享签约金额_T30,0) AS 飞享签约金额_T30
       ,NVL(v.飞跃签约金额_T30,0) AS 飞跃签约金额_T30
       ,NVL(v.提额签约金额_T30,0) AS 提额签约金额_T30
       ,NVL(v.飞享权益收入_T30,0) AS 飞享权益收入_T30
       ,NVL(v.飞跃权益收入_T30,0) AS 飞跃权益收入_T30
       ,NVL(v.提额权益收入_T30,0) AS 提额权益收入_T30
       ,NVL(v.总权益收入_T30,0)   AS 总权益收入_T30
       ,NVL(v.飞享签约金额_T60,0) AS 飞享签约金额_T60
       ,NVL(v.飞跃签约金额_T60,0) AS 飞跃签约金额_T60
       ,NVL(v.提额签约金额_T60,0) AS 提额签约金额_T60
       ,NVL(v.飞享权益收入_T60,0) AS 飞享权益收入_T60
       ,NVL(v.飞跃权益收入_T60,0) AS 飞跃权益收入_T60
       ,NVL(v.提额权益收入_T60,0) AS 提额权益收入_T60
       ,NVL(v.总权益收入_T60,0)   AS 总权益收入_T60
       ,NVL(v.飞享签约金额_T90,0) AS 飞享签约金额_T90
       ,NVL(v.飞跃签约金额_T90,0) AS 飞跃签约金额_T90
       ,NVL(v.提额签约金额_T90,0) AS 提额签约金额_T90
       ,NVL(v.飞享权益收入_T90,0) AS 飞享权益收入_T90
       ,NVL(v.飞跃权益收入_T90,0) AS 飞跃权益收入_T90
       ,NVL(v.提额权益收入_T90,0) AS 提额权益收入_T90
       ,NVL(v.总权益收入_T90,0)   AS 总权益收入_T90
FROM loan_agg l
LEFT JOIN vip_agg v
--ON l.曝光月 = v.曝光月 AND l.风险原始定价 = v.风险原始定价 AND l.vip_classify_group = v.vip_classify_group AND l.飞跃会员类型分组 = v.飞跃会员类型分组 AND l.group_tag = v.group_tag AND l.risk_I20 = v.risk_I20 AND l.额度区间 = v.额度区间 AND l.可用额度4k_10k = v.可用额度4k_10k;
ON l.曝光月 = v.曝光月 AND l.group_tag = v.group_tag AND l.可用额度4k_10k = v.可用额度4k_10k
'''

In [3]:
import query_analysis_tool as qat
loan_stats = qat.run_query(query)

# 字符串字段
str_cols = ['曝光月','风险原始定价','vip_classify_group','飞跃会员类型分组','group_tag','risk_I20','额度区间','可用额度4k_10k']

# 单元格15的 SQL 输出：贷款含 T0/T7/T30/T60/T90，会员卡含 T0/T7/T30/T60/T90
loan_windows = ['T0', 'T7', 'T30', 'T60', 'T90']
vip_windows = ['T0', 'T7', 'T30', 'T60', 'T90']

# 浮点数字段 - 保留2位小数
loan_float_metrics = ['放款金额', '定价', '期限', 'principal', 'annualized_interest_fee']
vip_cards = ['飞享', '飞跃', '提额']
vip_float_metrics = ['签约金额', '权益收入']

float_cols = [f'{metric}_{window}' for window in loan_windows for metric in loan_float_metrics]
float_cols += [f'{card}{metric}_{window}' for window in vip_windows for card in vip_cards for metric in vip_float_metrics]
float_cols += [f'总权益收入_{window}' for window in vip_windows]

# 整数字段
int_cols = ['首曝人数'] + [f'放款笔数_{window}' for window in loan_windows]

loan_stats = qat.format_dataframe_columns(
    loan_stats,
    str_cols=str_cols,
    date_cols=[], 
    int_cols=int_cols,
    float_cols=float_cols
)

qat.write_dataframe_to_excel(
    file_path=r"D:\4.临时取数\20+权益资产测试\消金20接资金预路由老客0529_full.xlsx",
    dataframes_dict={"T90观测数据": loan_stats},
    start_row=1,
    include_header=True
)

正在获取数据，首段 SQL: 
WITH exposure_base AS
(
	SELECT  *
	       ,SUBST ...
成功写入工作表: T90观测数据
文件已保存: D:\4.临时取数\20+权益资产测试\消金20接资金预路由老客0529_full.xlsx


In [ ]:
import query_analysis_tool as qat

windows = ['T0', 'T7', 'T30', 'T60', 'T90']
calc_fields = {}

for window in windows:
    calc_fields[f'{window}放款件均'] = {
        'formula': f"='放款金额_{window}'/'放款笔数_{window}'",
        'number_format': '0.00'
    }
    calc_fields[f'{window}加权期限'] = {
        'formula': f"='期限_{window}'/'放款金额_{window}'",
        'number_format': '0.00'
    }
    calc_fields[f'{window}加权定价'] = {
        'formula': f"='定价_{window}'/'放款金额_{window}'",
        'number_format': '0.00%'
    }
    calc_fields[f'{window}息费率'] = {
        'formula': f"='annualized_interest_fee_{window}'/'principal_{window}'",
        'number_format': '0.00%'
    }
    calc_fields[f'{window}权益收入占放款'] = {
        'formula': f"='总权益收入_{window}'/'放款金额_{window}'",
        'number_format': '0.00%'
    }

    for card in ['飞享', '飞跃', '提额']:
        display_card = '提额卡' if card == '提额' else card
        calc_fields[f'{window}{display_card}签约金额'] = {
            'formula': f"='{card}签约金额_{window}'/'放款金额_{window}'",
            'number_format': '0.00%'
        }
        calc_fields[f'{window}{display_card}权益收入'] = {
            'formula': f"='{card}权益收入_{window}'/'放款金额_{window}'",
            'number_format': '0.00%'
        }

qat.add_pivot_calculated_fields(
    file_path=r"D:\4.临时取数\20+权益资产测试\消金20接资金预路由老客0529_full.xlsx",
    sheet_name="summary",
    pivot_name="数据透视表18",
    fields=calc_fields
) 


[Pivot] sheet=summary pivot=数据透视表18 cache_index=8
[OK] Add CalculatedField: T0放款件均 | ='放款金额_T0'/'放款笔数_T0'
[OK] Add to Values: T0放款件均 | format=0.00
[OK] Add CalculatedField: T0加权期限 | ='期限_T0'/'放款金额_T0'
[OK] Add to Values: T0加权期限 | format=0.00
[OK] Add CalculatedField: T0加权定价 | ='定价_T0'/'放款金额_T0'
[OK] Add to Values: T0加权定价 | format=0.00%
[OK] Add CalculatedField: T0息费率 | ='annualized_interest_fee_T0'/'principal_T0'
[OK] Add to Values: T0息费率 | format=0.00%
[OK] Add CalculatedField: T0权益收入占放款 | ='总权益收入_T0'/'放款金额_T0'
[OK] Add to Values: T0权益收入占放款 | format=0.00%
[OK] Add CalculatedField: T0飞享签约金额 | ='飞享签约金额_T0'/'放款金额_T0'
[OK] Add to Values: T0飞享签约金额 | format=0.00%
[OK] Add CalculatedField: T0飞享权益收入 | ='飞享权益收入_T0'/'放款金额_T0'
[OK] Add to Values: T0飞享权益收入 | format=0.00%
[OK] Add CalculatedField: T0飞跃签约金额 | ='飞跃签约金额_T0'/'放款金额_T0'
[OK] Add to Values: T0飞跃签约金额 | format=0.00%
[OK] Add CalculatedField: T0飞跃权益收入 | ='飞跃权益收入_T0'/'放款金额_T0'
[OK] Add to Values: T0飞跃权益收入 | format=0.00%
[OK] Add CalculatedFi

[]